# Week 48, Day 6: Forge: Week 48 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
from scipy.stats import norm
print("1.", round(norm.ppf(0.9) - norm.ppf(0.1), 2))
print("2.", 1 / (1 + 1))
print("3.", 9 ** 0.5)

### Exercise: Part A: a likelihood-ratio test for a prior

Write nll(params, k_high, k_low, fixed_a=None): the summed binomial negative log-likelihood of both conditions, with P(test faster) = Phi((d + a (sig_t^2 - sig_h^2)) / sqrt(sig_t^2 + sig_h^2)), where sig_h = exp(params[0]), sig_l = exp(params[1]) and a = params[2] unless fixed. Fit both models to data simulated with a = -1.2 and with a = 0, and compute the likelihood ratio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, chi2
from scipy.optimize import minimize
r = np.random.default_rng(2002)
du = np.linspace(-0.6, 0.6, 9); n_tr = 80

def simulate(a, sh=0.15, sl=0.35):
    def p(d, st): return norm.cdf((d + a * (st ** 2 - sh ** 2)) / np.sqrt(st ** 2 + sh ** 2))
    return r.binomial(n_tr, p(du, sh)), r.binomial(n_tr, p(du, sl))

def nll(params, k_high, k_low, fixed_a=None):
    sh, sl = np.exp(params[0]), np.exp(params[1])
    a = params[2] if fixed_a is None else fixed_a
    total = 0.0
    for k, st in [(k_high, sh), (k_low, sl)]:
        p = ...
        total -= np.sum(k * np.log(p) + (n_tr - k) * np.log(1 - p))
    return total

lr = {}
for name, a_true in [("prior", -1.2), ("no prior", 0.0)]:
    kh, kl = simulate(a_true)
    full = minimize(nll, [np.log(0.2), np.log(0.3), 0.0], args=(kh, kl), method="Nelder-Mead", options={"maxiter": 4000})
    null = minimize(nll, [np.log(0.2), np.log(0.3)], args=(kh, kl, 0.0), method="Nelder-Mead")
    lr[name] = 2 * (null.fun - full.fun)
    print(f"{name:8s}: fitted a = {full.x[2]:+.2f}; likelihood ratio {lr[name]:.1f}, p = {chi2.sf(lr[name], 1):.2g}")

In [ ]:
import numpy as np
def test_detect():
    "with a true prior slope of -1.2, the likelihood ratio exceeds 10 (p well below 0.01)"
    assert lr["prior"] > 10
def test_null():
    "with no prior, the likelihood ratio is below 6.6 (not significant at 1%)"
    assert lr["no prior"] < 6.6

test_detect()
test_null()
print("All checks passed")

### Exercise: Part B: contrast-dependent slowing from spikes

Write perceived_speed(c): draw spikes with gain 20 c, compute the log-likelihood as logF @ r, add the log prior -s^2 / (2 sigma_p^2) with sigma_p = 8, normalise and return the posterior mean. Average over 200 trials for contrasts 0.05, 0.1, 0.3 and 1.

In [ ]:
pref = np.linspace(-40, 40, 61); s_grid = np.linspace(-20, 40, 1201); w = 6.0
logF = -(s_grid[:, None] - pref[None]) ** 2 / (2 * w ** 2)
sigma_p = 8.0

def perceived_speed(c, s_true=10.0):
    rs = r.poisson(20 * c * np.exp(-(s_true - pref) ** 2 / (2 * w ** 2)) + 0.05)
    lp = ...
    lp -= lp.max(); p = np.exp(lp); p /= p.sum()
    return (p * s_grid).sum()

perceived = {c: np.mean([perceived_speed(c) for _ in range(200)]) for c in [0.05, 0.1, 0.3, 1.0]}
for c, v in perceived.items():
    print(f"contrast {c:4.2f}: mean perceived speed {v:.2f} (true 10)")
plt.semilogx(list(perceived), list(perceived.values()), "o-"); plt.axhline(10, color="k", ls=":"); plt.xlabel("contrast"); plt.ylabel("perceived speed"); plt.show()

In [ ]:
import numpy as np
def test_slowing():
    "average perceived speed at contrast 0.05 is at least 20% below that at contrast 1"
    assert perceived[0.05] < 0.8 * perceived[1.0]
def test_high():
    "at full contrast the perceived speed is within 10% of the true speed"
    assert abs(perceived[1.0] - 10) < 1.0

test_slowing()
test_high()
print("All checks passed")